# Poisson SLDS with switching rotational dynamics

This example uses a two-dimensional continuous latent trajectory with two discrete
dynamical states: clockwise and counter-clockwise rotation. A shared population of
Poisson neurons reads out the same latent space in both states, so the discrete state
is identified by how the latent trajectory moves rather than by where it is.


In [ ]:
%matplotlib inline

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import posterior_plots

from xxm import DatasetCollection, Sequences, plot
from xxm.core.align import align_procrustes as align_latent
from xxm.core.align import match_states_to_true as match_states
from xxm.slds import PoissonSLDS, SLDSParams

In [ ]:
NUM_NEURONS = 12
NUM_STEPS = 1_500


def make_rotation(angle: float, radius: float) -> jax.Array:
    """Return a damped 2D rotation matrix."""
    return radius * jnp.array(
        [
            [jnp.cos(angle), -jnp.sin(angle)],
            [jnp.sin(angle), jnp.cos(angle)],
        ]
    )


def make_emission_coefficients(
    num_neurons: int,
    scale: float,
) -> jax.Array:
    """Arrange Poisson readout directions uniformly around the latent plane."""
    preferred_angles = jnp.linspace(
        0.0,
        2.0 * jnp.pi,
        num_neurons,
        endpoint=False,
    )

    return scale * jnp.stack(
        [
            jnp.cos(preferred_angles),
            jnp.sin(preferred_angles),
        ],
        axis=1,
    )


def make_true_model(
    *,
    num_neurons: int,
    angle: float = 0.18,
    radius: float = 0.995,
    self_transition_prob: float = 0.98,
    initial_radius: float = 1.5,
    initial_variance: float = 0.05,
    dynamics_variance: float = 0.01,
    emission_scale: float = 0.45,
    base_rate: float = 4.0,
) -> PoissonSLDS:
    """Build a two-state Poisson SLDS with opposing rotational dynamics."""
    dynamics_coefficients = jnp.stack(
        [
            make_rotation(-angle, radius),
            make_rotation(angle, radius),
        ]
    )

    num_states = dynamics_coefficients.shape[0]
    latent_dim = dynamics_coefficients.shape[-1]
    switch_prob = 1.0 - self_transition_prob

    emission_coefficients = make_emission_coefficients(
        num_neurons=num_neurons,
        scale=emission_scale,
    )

    return PoissonSLDS.from_params(
        SLDSParams(
            initial_probs=jnp.full(num_states, 1.0 / num_states),
            transition_probs=jnp.array(
                [
                    [self_transition_prob, switch_prob],
                    [switch_prob, self_transition_prob],
                ]
            ),
            latent_initial_means=jnp.tile(
                jnp.array([initial_radius, 0.0]),
                (num_states, 1),
            ),
            latent_initial_covariances=jnp.tile(
                initial_variance * jnp.eye(latent_dim),
                (num_states, 1, 1),
            ),
            dynamics_coefficients=dynamics_coefficients,
            dynamics_bias=jnp.zeros((num_states, latent_dim)),
            dynamics_covariances=jnp.tile(
                dynamics_variance * jnp.eye(latent_dim),
                (num_states, 1, 1),
            ),
            emissions=(
                {
                    'coefficients': emission_coefficients,
                    'bias': jnp.full(
                        num_neurons,
                        jnp.log(base_rate),
                    ),
                },
            ),
        )
    )


true_model = make_true_model(
    num_neurons=NUM_NEURONS,
)

truth = true_model.sample(
    key=jax.random.key(0),
    steps=NUM_STEPS,
)

data = DatasetCollection.from_array(truth.observations.get())

In [ ]:
def plot_data(
    states,
    latents,
    observations,
) -> None:
    _, axs = plt.subplots(
        nrows=2,
        sharex='all',
        figsize=(8, 5),
        constrained_layout=True,
        gridspec_kw={'height_ratios': [1, 2]},
    )

    plot.plot_seq_1d(
        axs[0],
        states,
        latents,
        desc='latent',
    )
    axs[0].set(title='Latent trajectory')

    axs[1].figure.colorbar(
        plot.plot_traces_image(axs[1], observations, ylabel='neuron'),
        ax=axs[1],
        label='count',
    )
    axs[1].set(title='Poisson observations')


plot_data(
    truth.states.get(),
    truth.latents.get(),
    truth.observations.get(),
)

In [ ]:
_, ax = plt.subplots(
    figsize=(4, 4),
    constrained_layout=True,
)

plot.plot_seq_2d(
    ax,
    truth.states.get(),
    truth.latents.get(),
    desc='latent',
)
ax.set(title='True latent trajectory')

plot.plot_dyn_conditional_linear(true_model.dynamics())

In [ ]:
inferred = true_model.infer(data, num_iters=100)

oracle_states = inferred.most_likely_states()
oracle_latents = inferred.latent_posterior().mean


plot.plot_seq_1d_comparison(
    truth.states.get(),
    truth.latents.get(),
    oracle_states.get(),
    oracle_latents.get(),
)

In [ ]:
initial_model = PoissonSLDS.via_arhmm(
    key=jax.random.key(1),
    data=data,
    num_states=true_model.num_states,
    latent_dim=true_model.latent_dim,
    num_arhmm_iters=1000,
).get()

fitted, objective_trace = initial_model.fit(
    data,
    num_iters=300,
    num_inference_iters=5,
)


plot.plot_fit_progress(
    objective_trace,
    name='ELBO',
    title='Laplace EM',
)

In [ ]:
def align_model(
    fitted,
    true_states: Sequences,
    true_latents: Sequences,
):

    permutation = match_states(
        fitted.state_probs().get(),
        true_states.get(),
    )

    fitted = fitted.permute_states(permutation)

    alignment = align_latent(
        fitted.latent_posterior().mean.get(),
        true_latents.get(),
    )

    fitted = fitted.align(alignment)

    return fitted


fitted = align_model(
    fitted,
    truth.states,
    truth.latents,
)

In [ ]:
plot.plot_seq_1d_comparison(
    truth.states.get(),
    truth.latents.get(),
    fitted.most_likely_states().get(),
    fitted.latent_posterior().mean.get(),
)

plot.plot_seq_2d_comparison(
    truth.states.get(),
    truth.latents.get(),
    fitted.most_likely_states().get(),
    fitted.latent_posterior().mean.get(),
)

latent_posterior = fitted.latent_posterior().get()

posterior_plots.plot_posterior_over_time(latent_posterior, truth.latents.get())
posterior_plots.plot_occupancy(latent_posterior, truth.latents.get())

In [ ]:
plot.plot_traces_image_comparison(
    truth.observations.get(),
    fitted.observation_mean().get(),
    ylabel='neuron',
    name0='observations',
    name1='posterior mean rates',
    desc='count',
)

In [ ]:
plot.plot_dyn_conditional_linear_comparison(
    true_model.dynamics(),
    fitted.model.dynamics(),
)